# Fog-IDS: Resource-Aware Multi-Class Intrusion Detection for Fog Computing Nodes
### End-to-End Interactive System Walkthrough & Research Demonstration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raghunath2604/RIO_PROJECT-SMART_CITY-/blob/main/notebooks/Fog_IDS_Walkthrough.ipynb)
![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-22c55e.svg)
![Dataset](https://img.shields.io/badge/Dataset-CICIoT2023-ffffff.svg)

**Author:** Raghunatha Reddy G R (SRN: R23EA094)  
**Institution:** School of Computing Science and Engineering, REVA University, Bengaluru, India  
**Faculty Supervisor:** Dr. Supreeth S  

---
## 1. System Overview
This notebook provides an interactive walkthrough of **Fog-IDS**, a resource-aware intrusion detection appliance designed for Smart City edge gateways and municipal fog clusters.

### Key Capabilities Demonstrated:
1. **Model Zoo & Checksums**: Auditing 12 production models across Binary, 8-Class, and 34-Class tasks.
2. **Real Flow Ingestion**: Classifying genuine CICIoT2023 network flows with sub-millisecond latencies.
3. **Explainable AI (XAI)**: Local feature attribution identifying attack vectors.
4. **FR5 Dynamic Tier Switching**: Closed-loop CPU headroom monitoring with deadband hysteresis (ΔH = 5%) and dwell-time (τ = 1.0s).

In [ ]:
# Setup environment and repository paths
import os, sys

# If running in Google Colab, clone repository
if 'google.colab' in sys.modules:
    !git clone https://github.com/Raghunath2604/RIO_PROJECT-SMART_CITY-.git
    %cd RIO_PROJECT-SMART_CITY-
    !pip install -r requirements.txt

sys.path.insert(0, os.getcwd())
print(f"[READY] Working Directory: {os.getcwd()}")

## 2. Model Integrity Verification (SHA-256 Checksums)
Every model artifact is cryptographically validated against `models/manifest.json` before loading into edge memory.

In [ ]:
from models.model_loader import verify_all_artifacts, load_manifest

manifest = load_manifest()
audit = verify_all_artifacts()

print(f"Verification Status: {'SUCCESS' if audit['valid'] else 'FAILED'}")
print(f"Intact Models:       {audit['verified_models']} / {audit['total_models']}")
for k, v in list(audit['details'].items())[:4]:
    print(f"  • {k:<25} -> {v['status']} | SHA256: {v.get('sha256', '')[:12]}...")

## 3. Real-Time Packet Flow Ingestion & Threat Classification
Classifying realistic network flows across multiple threat categories (DDoS, Mirai, Brute Force, Benign IoT).

In [ ]:
from src.infer import FogInferenceEngine, get_preset_attack_samples

engine = FogInferenceEngine()
presets = get_preset_attack_samples()

print(f"{'TEST FLOW':<36} | {'PREDICTION':<16} | {'SEVERITY':<10} | {'CONFIDENCE':<10} | {'LATENCY (µs)':<12}")
print("-" * 92)

for flow_name, features in presets.items():
    res = engine.predict_sample(features, model_name="LightGBM", task="8class")
    pred = res["prediction"]
    sev = res["severity"]
    conf = res["confidence"] * 100
    lat = res["latency_us"]
    print(f"{flow_name:<36} | {pred:<16} | {sev:<10} | {conf:>8.1f}% | {lat:>10.1f} µs")

## 4. Explainable AI (XAI) Local Feature Attribution
Providing explainability by calculating feature impacts (positive attack indicators vs. normalizing factors).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.explain import explain_prediction
from src.model_registry import ModelRegistry
from src.schema import FEATURES

reg = ModelRegistry()
bundle = reg.load_model("LightGBM", "8class")
ddos_sample = presets["DDoS-SYN_Flood (Critical)"]
vec = np.array([ddos_sample.get(f, 0.0) for f in FEATURES], dtype=np.float64)

attributions = explain_prediction(bundle["estimator"], bundle["scaler"], bundle["label_encoder"], vec)

top_attrs = attributions[:8]
feat_names = [a["feature"] for a in top_attrs][::-1]
scores = [a["impact_score"] for a in top_attrs][::-1]
colors = ['#4ade80' if s > 0 else '#ef4444' for s in scores]

plt.figure(figsize=(10, 4.5), dpi=120)
plt.barh(feat_names, scores, color=colors)
plt.title("Top 8 Local Feature Attributions (XAI Impact) - DDoS Flow", fontsize=12, fontweight='bold')
plt.xlabel("Attribution Impact Score (+ = Attack Indicator, - = Normalizing)")
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. FR5 Dynamic Resource-Aware Tier Switching
Simulating CPU load spikes (20% -> 60% -> 90% -> 35%) to verify asymmetric deadband hysteresis (ΔH = 5%) and dwell-time smoothing (τ = 1.0s).

In [ ]:
from src.tier_switcher import DynamicTierSwitcher

switcher = DynamicTierSwitcher(dwell_time_seconds=0.0, simulate_resources=True)
load_scenarios = [
    ("Normal Baseline", 25.0),
    ("Elevated Traffic Surge", 55.0),
    ("Extreme Resource Spike", 88.0),
    ("Recovery (Boundary Hysteresis)", 72.0),
    ("Full Recovery", 30.0)
]

print(f"{'SCENARIO':<32} | {'CPU LOAD':<10} | {'ACTIVE TIER':<14} | {'SELECTED MODEL':<14}")
print("-" * 76)

for name, cpu in load_scenarios:
    switcher.set_simulated_load(cpu, 50.0)
    res = switcher.predict(np.zeros((1, 46)), task="8class")
    print(f"{name:<32} | {cpu:>8.1f}% | {res['tier']:<14} | {res['model_name']:<14}")

## 6. High-Throughput Batch Inference Benchmark
Measuring raw vectorized inference throughput on genuine test batches.

In [ ]:
import time

# Load real samples from test dataset
csv_path = "datasets/sample_test.csv"
if os.path.exists(csv_path):
    df_test = pd.read_csv(csv_path, nrows=500)
    feature_cols = [c for c in FEATURES if c in df_test.columns]
    X_matrix = df_test[feature_cols].values

    # Profile CompactMLP (Reduced Tier)
    bundle_mlp = reg.load_model("CompactMLP", "8class")
    X_scaled = bundle_mlp["scaler"].transform(X_matrix)

    t0 = time.perf_counter()
    preds = bundle_mlp["estimator"].predict(X_scaled)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    print(f"Batch Size:               {len(X_matrix)} flows")
    print(f"Total Execution Time:     {elapsed_ms:.2f} ms")
    print(f"Per-Flow Ingestion Time:  {(elapsed_ms * 1000 / len(X_matrix)):.2f} µs/flow")
    print(f"Throughput:               {(len(X_matrix) / (elapsed_ms / 1000)):.1f} flows/sec")
else:
    print(f"Dataset file {csv_path} not found.")

## 7. Conclusion & Research Artifacts
- **Live Web Console**: [https://rio-project-smart-city.vercel.app/](https://rio-project-smart-city.vercel.app/)
- **GitHub Repository**: [https://github.com/Raghunath2604/RIO_PROJECT-SMART_CITY-](https://github.com/Raghunath2604/RIO_PROJECT-SMART_CITY-)
- **Full Documentation**: See `docs/PRD.md`, `docs/TRD.md`, and `docs/RUNBOOK.md` in the repository.